# 03 — Inference (No-CoT variant)

Single-example exploration of the no-CoT LoRA adapter trained by `02_sft_training_nocot_0511.py`. Mirrors the training-time prompt construction exactly (same `SYSTEM_CONTENT` / `USER_TMPL`, 4 inputs: transcript + previous_call_summary + current_call_summary + objection_summary).

**Standalone:** this notebook only depends on `03_inference_nocot.py` (and through it, the training script for prompt templates). It does **not** import from `03_inference.py`, which targets the teacher-CoT variant and has a different prompt schema (history_summary + window_size).

**Central sanity check for the nocot variant:** the analysis channel is *not* trained on (its span is empty in training). At inference, the base reasoning prior is expected to still produce content in the analysis channel — this notebook prints both channels so you can verify that survival.

Works with either the final adapter (`checkpoints_nocot/final_adapter/`) or any intermediate checkpoint (`checkpoints_nocot/checkpoint-N/`). Memory: bf16 120B ≈ 240 GB across visible GPUs.

Batch over a DataFrame: run `03_inference_nocot.py`.

## Cell 0 — GPU selection

Must run **before** any `torch` import so `device_map="auto"` only sees the GPUs we want.

In [ ]:
import os
VISIBLE_GPUS = [0, 1, 2, 3, 4, 5]
assert "torch" not in globals(), "restart kernel before changing VISIBLE_GPUS"
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in VISIBLE_GPUS)
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## Cell 1 — Imports and config

Load the sibling `03_inference_nocot.py` (filename starts with a digit, so we use `importlib.util` rather than a plain `import`). Everything we need — `load_model_with_adapter`, `generate_response`, `parse_channels`, prompt templates — is re-exported from that module.

In [ ]:
import importlib.util
from pathlib import Path

def _load(name, filename):
    spec = importlib.util.spec_from_file_location(name, filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

infer = _load("infer_nocot", "03_inference_nocot.py")

MODEL_DIR = "/path/to/gpt-oss-120b"                          # <-- FILL IN
ADAPTER_PATH = "checkpoints_nocot/final_adapter"             # final OR checkpoint-N
REASONING_EFFORT = "medium"                                  # low / medium / high
MAX_NEW_TOKENS = 1024
MAX_TRANSCRIPT_CHARS = infer.DEFAULT_MAX_TRANSCRIPT_CHARS    # same as training

## Cell 2 — Load base model + LoRA adapter

PEFT accepts any directory with `adapter_config.json` + the adapter weights file, so `ADAPTER_PATH` can point at `checkpoints_nocot/final_adapter` OR any intermediate `checkpoints_nocot/checkpoint-N`.

In [ ]:
model, tokenizer = infer.load_model_with_adapter(MODEL_DIR, ADAPTER_PATH)
print(f"Loaded. Device map: {getattr(model, 'hf_device_map', '<single>')}")
print(f"Adapter source:    {ADAPTER_PATH}")

## Cell 3 — `generate_response(...)` — the function you asked for

Thin wrapper that closes over `model` and `tokenizer` so callers only pass the 4 inputs (plus optional `customer_context`). Returns the raw decode and the parsed `analysis` and `final` channels. Set `include_base=True` to also run with the adapter disabled for an A/B against the base model.

In [ ]:
def generate_response(
    transcript: str,
    previous_call_summary: str,
    current_call_summary: str,
    objection_summary: str,
    customer_context: str = "",
    reasoning_effort: str = REASONING_EFFORT,
    max_new_tokens: int = MAX_NEW_TOKENS,
    max_transcript_chars: int = MAX_TRANSCRIPT_CHARS,
    include_base: bool = False,
) -> dict:
    """Single-shot generation matching training-time prompt construction.

    Returns {'raw', 'analysis', 'final'}, optionally plus
    {'base_raw', 'base_analysis', 'base_final'} when include_base=True.
    """
    return infer.generate_response(
        model, tokenizer,
        transcript=transcript,
        previous_call_summary=previous_call_summary,
        current_call_summary=current_call_summary,
        objection_summary=objection_summary,
        customer_context=customer_context,
        reasoning_effort=reasoning_effort,
        max_new_tokens=max_new_tokens,
        max_transcript_chars=max_transcript_chars,
        include_base=include_base,
    )

## Cell 4 — Sample inputs + run

Replace the four strings with whatever you want to test.

In [ ]:
TRANSCRIPT = """agent: Hi, this is Sarah from American Express. Am I speaking with Mr. Johnson?
customer: Yes, this is he. What is this about?
agent: I'm calling about a pre-approved offer for our Platinum card with a $200 statement credit after your first purchase.
customer: I already have a Platinum card. I've had it for years."""

PREVIOUS_CALL_SUMMARY = "No prior contact with this customer on record."

CURRENT_CALL_SUMMARY = (
    "Agent introduced themselves and the Platinum card pre-approved offer "
    "with a $200 statement credit on first purchase."
)

OBJECTION_SUMMARY = (
    "Customer says they already have a Platinum card and have had it for years — "
    "implicit objection: why would they want another one."
)

CUSTOMER_CONTEXT = ""  # populate if your get_customer_context() reads a column

result = generate_response(
    transcript=TRANSCRIPT,
    previous_call_summary=PREVIOUS_CALL_SUMMARY,
    current_call_summary=CURRENT_CALL_SUMMARY,
    objection_summary=OBJECTION_SUMMARY,
    customer_context=CUSTOMER_CONTEXT,
)

print("=" * 80)
print("ANALYSIS (CoT — not trained, base prior)")
print("=" * 80)
print(result["analysis"] or "<EMPTY>")
print()
print("=" * 80)
print("FINAL (trained agent response)")
print("=" * 80)
print(result["final"] or "<EMPTY>")
print()
n_a = len(tokenizer.encode(result["analysis"], add_special_tokens=False))
n_f = len(tokenizer.encode(result["final"], add_special_tokens=False))
print(f"analysis tokens: {n_a}   final tokens: {n_f}")
assert n_f > 0, "REGRESSION: final channel is empty — adapter / prompt is broken."
if n_a == 0:
    print("NOTE: analysis is empty. Expected non-empty for the nocot variant if "
          "the base reasoning prior survived SFT — investigate.")

## Cell 5 — A/B against base model (adapter disabled)

Confirms the adapter is actually being applied — final responses should differ.

In [ ]:
ab = generate_response(
    transcript=TRANSCRIPT,
    previous_call_summary=PREVIOUS_CALL_SUMMARY,
    current_call_summary=CURRENT_CALL_SUMMARY,
    objection_summary=OBJECTION_SUMMARY,
    customer_context=CUSTOMER_CONTEXT,
    include_base=True,
)

for label, a_key, f_key in [
    ("ADAPTER", "analysis", "final"),
    ("BASE   ", "base_analysis", "base_final"),
]:
    print("=" * 80)
    print(f"{label} — analysis")
    print("=" * 80)
    print((ab[a_key] or "<EMPTY>")[:1500])
    print()
    print(f"{label} — final")
    print("-" * 80)
    print(ab[f_key] or "<EMPTY>")
    print()

print("Adapter vs base final differ:", ab["final"] != ab["base_final"])

## Cell 6 — Sweep `reasoning_effort` ∈ {low, medium, high}

No model reload needed — the chat template re-renders the system content with the new effort tag.

In [ ]:
for effort in ["low", "medium", "high"]:
    r = generate_response(
        transcript=TRANSCRIPT,
        previous_call_summary=PREVIOUS_CALL_SUMMARY,
        current_call_summary=CURRENT_CALL_SUMMARY,
        objection_summary=OBJECTION_SUMMARY,
        customer_context=CUSTOMER_CONTEXT,
        reasoning_effort=effort,
    )
    n_a = len(tokenizer.encode(r["analysis"], add_special_tokens=False))
    n_f = len(tokenizer.encode(r["final"], add_special_tokens=False))
    print(f"effort={effort:<6} analysis_tokens={n_a:>5}  final_tokens={n_f:>5}")

## Cell 7 — Interactive chat form (`ipywidgets`)

Five textareas for the 4 inputs (+ optional customer_context), an effort dropdown, an A/B checkbox, and a Generate button. Each click is independent — no persistent multi-turn state (the model is trained single-shot, so a real chat loop would be off-distribution).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

ta_transcript = widgets.Textarea(
    description="transcript", layout=widgets.Layout(width="100%", height="220px"),
    style={"description_width": "180px"},
)
ta_prev = widgets.Textarea(
    description="previous_call_summary", layout=widgets.Layout(width="100%", height="80px"),
    style={"description_width": "180px"},
)
ta_curr = widgets.Textarea(
    description="current_call_summary", layout=widgets.Layout(width="100%", height="80px"),
    style={"description_width": "180px"},
)
ta_obj = widgets.Textarea(
    description="objection_summary", layout=widgets.Layout(width="100%", height="80px"),
    style={"description_width": "180px"},
)
ta_ctx = widgets.Textarea(
    description="customer_context", layout=widgets.Layout(width="100%", height="60px"),
    style={"description_width": "180px"},
)
dd_effort = widgets.Dropdown(
    description="reasoning_effort", options=["low", "medium", "high"],
    value=REASONING_EFFORT, style={"description_width": "180px"},
)
cb_base = widgets.Checkbox(description="Include base-model A/B", value=False)
btn = widgets.Button(description="Generate", button_style="primary")
out = widgets.Output()

# Pre-fill from Cell 4 if available.
ta_transcript.value = TRANSCRIPT
ta_prev.value = PREVIOUS_CALL_SUMMARY
ta_curr.value = CURRENT_CALL_SUMMARY
ta_obj.value = OBJECTION_SUMMARY
ta_ctx.value = CUSTOMER_CONTEXT

def _on_click(_):
    with out:
        clear_output()
        print("Generating…")
        r = generate_response(
            transcript=ta_transcript.value,
            previous_call_summary=ta_prev.value,
            current_call_summary=ta_curr.value,
            objection_summary=ta_obj.value,
            customer_context=ta_ctx.value,
            reasoning_effort=dd_effort.value,
            include_base=cb_base.value,
        )
        clear_output()
        display(Markdown("### Adapter — analysis"))
        display(Markdown("```\n" + (r["analysis"] or "<EMPTY>") + "\n```"))
        display(Markdown("### Adapter — final"))
        display(Markdown("```\n" + (r["final"] or "<EMPTY>") + "\n```"))
        if cb_base.value:
            display(Markdown("### Base — analysis"))
            display(Markdown("```\n" + (r["base_analysis"] or "<EMPTY>") + "\n```"))
            display(Markdown("### Base — final"))
            display(Markdown("```\n" + (r["base_final"] or "<EMPTY>") + "\n```"))

btn.on_click(_on_click)
display(widgets.VBox([ta_transcript, ta_prev, ta_curr, ta_obj, ta_ctx,
                      widgets.HBox([dd_effort, cb_base, btn]), out]))